[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C40_Research_Methodology_Course/04_research_engineering/04_research_engineering.ipynb)

# 04 · 研究工程卫生（动手做）

目标：把研究工程跑成代码——**轻量 config 系统**、**网格 vs 随机 sweep**、**可复现(固定种子+逐位核验)**、**结果聚合**。

路线：config 系统(默认+覆盖+序列化) → 网格 sweep → 随机 sweep → 可复现核验 → pandas 聚合 → ✏️ 练习 → 📖 答案 → 🧪 完整实验管线胶囊。

> 核心心法：**每个结果都应能用『config + 种子』完全重建。** 务实，不求完美。

## 1 · worked：轻量 config 系统(默认值 + 覆盖 + 序列化)

把参数从代码揪出来：一份带默认值的 config，可被覆盖、可存读 JSON。这是 sweep 与可复现的基础。

In [ ]:
import numpy as np
import pandas as pd
import json

DEFAULT_CONFIG = dict(lr=0.01, batch=32, seed=0, model='mlp', n_layers=2)

def make_config(**overrides):
    '''从默认值出发，用 overrides 覆盖个别项，返回完整 config。'''
    cfg = dict(DEFAULT_CONFIG)          # 拷贝默认
    for k, v in overrides.items():
        assert k in DEFAULT_CONFIG, f'未知参数 {k}（防止拼写错误悄悄失效）'
        cfg[k] = v
    return cfg

def save_config(cfg, path):
    with open(path, 'w') as f: json.dump(cfg, f, sort_keys=True)
def load_config(path):
    with open(path) as f: return json.load(f)

cfg = make_config(lr=0.001, seed=42)
print('默认 + 覆盖(lr,seed) =', cfg)
assert cfg['lr'] == 0.001 and cfg['seed'] == 42
assert cfg['batch'] == 32, '未覆盖的项应保留默认'
# 序列化往返
import tempfile, os
path = os.path.join(tempfile.gettempdir(), 'cfg.json')
save_config(cfg, path)
assert load_config(path) == cfg, 'config 存读应往返一致(可存档可复现)'
# 拼写错误被挡住
try:
    make_config(learnig_rate=0.1); raise AssertionError('应报错')
except AssertionError as e:
    assert 'learnig_rate' in str(e)
print('✅ config 系统：默认值+覆盖+序列化+拼写校验，全部就绪。')

## 2 · worked：一个可复现的『实验』

用纯 numpy 玩具目标函数模拟『按 config 训练并评测』。关键：**给定 config，结果完全确定**(固定种子)。

In [ ]:
def run_experiment(cfg):
    '''玩具实验：合成目标函数，最优 lr≈0.01。返回 (score) + 种子噪声。
       给定相同 cfg(含 seed)，结果完全可复现。'''
    rng = np.random.default_rng(cfg['seed'])
    # 真实关系：score 在 lr=0.01 处最高，偏离则下降；层数适中更好
    lr_term = -((np.log10(cfg['lr']) - np.log10(0.01)) ** 2) * 0.05
    depth_term = -abs(cfg['n_layers'] - 3) * 0.01
    noise = rng.normal(0, 0.005)        # 种子方差
    return 0.90 + lr_term + depth_term + noise

cfg = make_config(lr=0.01, n_layers=3, seed=0)
r1 = run_experiment(cfg)
r2 = run_experiment(make_config(lr=0.01, n_layers=3, seed=0))   # 同 config 再跑
print(f'同一 config 两次运行: {r1:.6f}  vs  {r2:.6f}')
assert r1 == r2, '可复现核验：同 config(同种子)必须逐位相同！'
# 换种子应不同(有随机性)，但换回原种子又相同
assert run_experiment(make_config(seed=1)) != r1
print('✅ 可复现：同 config 逐位相同；换种子才变 -> 随机源被正确固定。')

## 3 · worked：网格 sweep

枚举超参的所有组合。直观但组合数指数爆炸，且在不重要维度上浪费预算。

In [ ]:
import itertools

def grid_sweep(grid):
    '''grid: {参数名: [候选值,...]}。返回所有组合的 config 覆盖项列表。'''
    keys = list(grid.keys())
    combos = []
    for values in itertools.product(*[grid[k] for k in keys]):
        combos.append(dict(zip(keys, values)))
    return combos

grid = {'lr': [0.1, 0.01, 0.001], 'n_layers': [2, 3, 4]}
combos = grid_sweep(grid)
print(f'网格大小 = {len(combos)} 个组合 (3×3)')
assert len(combos) == 9

# 跑网格，记录每个组合的 score
rows = []
for ov in combos:
    cfg = make_config(**ov, seed=0)
    rows.append(dict(**ov, score=run_experiment(cfg)))
grid_df = pd.DataFrame(rows).sort_values('score', ascending=False)
print(grid_df.head(3).to_string(index=False))
best = grid_df.iloc[0]
assert abs(best['lr'] - 0.01) < 1e-9, '网格应找到最优 lr=0.01'
print(f"✅ 网格搜索找到最优: lr={best['lr']}, n_layers={best['n_layers']}")

## 4 · worked：随机 sweep，且常优于网格

从分布采样固定预算个点。Bergstra & Bengio：少数超参重要时，随机搜索在重要维度上覆盖更多值 -> 同预算下更优。
我们造一个『只有 lr 重要、n_layers 几乎不重要』的场景验证。

In [ ]:
def run_lr_dominated(cfg):
    '''只有 lr 真正重要(最优 0.01)，n_layers 几乎无影响。'''
    rng = np.random.default_rng(cfg['seed'])
    lr_term = -((np.log10(cfg['lr']) - np.log10(0.01)) ** 2) * 0.1
    depth_term = -abs(cfg['n_layers'] - 3) * 0.0005   # 影响极小
    return 0.90 + lr_term + depth_term + rng.normal(0, 0.001)

BUDGET = 9
# 网格: lr 取 3 个『没押中最优 0.01』的值(现实中你很难恰好猜中最优)，n_layers 3 个 -> 3×3=9
grid_combos = grid_sweep({'lr':[0.05, 0.005, 0.0005], 'n_layers':[2,3,4]})
grid_best = max(run_lr_dominated(make_config(**c, seed=0)) for c in grid_combos)

# 随机: 9 个点，lr 从对数均匀采样(9 个不同值，更可能撞到最优 0.01 附近)
def random_sweep(budget, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(budget):
        lr = 10 ** rng.uniform(-3, -1)          # 对数均匀
        n_layers = int(rng.integers(2, 5))
        out.append(dict(lr=lr, n_layers=n_layers))
    return out

# 随机搜索有随机性：用『多次随机 sweep 的平均最优』来做公平的统计比较(Bergstra & Bengio 的命题是期望意义上的)
rand_bests = []
for s in range(200):
    combos = random_sweep(BUDGET, seed=s)
    rand_bests.append(max(run_lr_dominated(make_config(**c, seed=0)) for c in combos))
rand_best_mean = np.mean(rand_bests)

print(f'同预算 {BUDGET} 次:')
print(f'  网格最优 score        = {grid_best:.5f}  (lr 只试了 3 个固定值，未押中最优)')
print(f'  随机最优 score(平均)  = {rand_best_mean:.5f}  (lr 每次试 {BUDGET} 个不同值)')
print(f'  随机胜过网格的比例    = {np.mean([r > grid_best for r in rand_bests]):.0%}')
# 当最优不在网格上、且只有少数超参重要时，随机搜索期望上更优
assert rand_best_mean > grid_best, '随机搜索(平均)应优于未押中最优的网格'
print('✅ 最优不在网格上时，随机搜索同预算下期望更优(在重要维度 lr 上覆盖更密)。')

## 5 · worked：结果聚合(多种子 -> 均值±方差表)

把多 config × 多种子的运行存成长表，用 pandas groupby 聚合成可读的『每配置 均值±std±count』。

In [ ]:
# 跑 2 个配置 × 5 个种子
records = []
for lr in [0.01, 0.001]:
    for seed in range(5):
        cfg = make_config(lr=lr, n_layers=3, seed=seed)
        records.append(dict(lr=lr, seed=seed, score=run_experiment(cfg)))
df = pd.DataFrame(records)

# 聚合：按 config(lr) 分组，算 均值/标准差/运行数
summary = (df.groupby('lr')['score']
             .agg(['mean', 'std', 'count'])
             .reset_index()
             .sort_values('mean', ascending=False))
print(summary.to_string(index=False))
assert (summary['count'] == 5).all(), '每个配置应有 5 个种子'
assert summary.iloc[0]['lr'] == 0.01, '最优 lr=0.01 应均值最高'
# 报均值必带方差(模块03)：std 列就是画误差棒的来源(模块05)
assert 'std' in summary.columns
print('✅ groupby+agg 把多种子运行聚合成带方差的表 -> 直接喂给诚实图表的误差棒。')

---
## ✏️ 练习 1：config 覆盖与合并

实现 `merge_configs(base, override)`：返回一个新 dict = base 被 override 覆盖（不修改原 dict）。
override 里 base 没有的键也要加进去。

In [ ]:
def merge_configs(base, override):
    # TODO: 返回新 dict：先拷贝 base，再用 override 更新；不修改入参
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
base = dict(lr=0.01, batch=32)
merged = merge_configs(base, dict(lr=0.1, seed=5))
assert merged == dict(lr=0.1, batch=32, seed=5)
assert base == dict(lr=0.01, batch=32), '不应修改原 base（无副作用）'
print('✅ 练习 1 通过：config 合并正确且无副作用')

## ✏️ 练习 2：随机 sweep 采样器

实现 `sample_configs(space, n, seed)`：从搜索空间随机采样 `n` 个 config。
`space` 形如 `{'lr': ('loguniform', 1e-4, 1e-1), 'batch': ('choice', [16,32,64])}`。
`loguniform` 在对数尺度均匀采样；`choice` 从列表等概率选一个。

In [ ]:
def sample_configs(space, n, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        cfg = {}
        for name, spec in space.items():
            kind = spec[0]
            # TODO: kind=='loguniform': lo,hi=spec[1],spec[2]; 10**rng.uniform(log10(lo),log10(hi))
            #       kind=='choice':     从 spec[1] 列表里 rng.choice 选一个
            raise NotImplementedError
        out.append(cfg)
    return out

In [ ]:
# —— 练习 2 自测 ——
space = {'lr': ('loguniform', 1e-4, 1e-1), 'batch': ('choice', [16, 32, 64])}
cfgs = sample_configs(space, 50, seed=0)
assert len(cfgs) == 50
assert all(1e-4 <= c['lr'] <= 1e-1 for c in cfgs), 'lr 应落在范围内'
assert all(c['batch'] in [16,32,64] for c in cfgs), 'batch 应来自候选'
# 对数均匀：约一半样本 lr < 几何中点 1e-2.5
mid = 10 ** ((np.log10(1e-4) + np.log10(1e-1)) / 2)
frac_below = np.mean([c['lr'] < mid for c in cfgs])
assert 0.3 < frac_below < 0.7, 'loguniform 应在对数尺度均匀(约一半在几何中点下)'
print('✅ 练习 2 通过：随机 sweep 采样器(loguniform + choice)正确')

## ✏️ 练习 3：可复现自检

实现 `is_reproducible(run_fn, cfg)`：用同一个 cfg 调用 `run_fn` 两次，返回两次结果**是否完全相同**。
这是判断『一个实验是否可复现』的最直接核验。

In [ ]:
def is_reproducible(run_fn, cfg):
    # TODO: 用 dict(cfg) 拷贝两次分别调用 run_fn，返回两次结果是否 ==
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
good_cfg = make_config(seed=0)
assert is_reproducible(run_experiment, good_cfg) == True, '固定种子的实验应可复现'
# 一个『不可复现』的函数：每次用系统熵
def bad_run(cfg):
    return np.random.default_rng().normal()    # 没用 cfg['seed']！
assert is_reproducible(bad_run, good_cfg) == False, '不读种子的实验不可复现'
print('✅ 练习 3 通过：能自动核验一个实验是否可复现')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def merge_configs(base, override):
    merged = dict(base)
    merged.update(override)
    return merged

In [ ]:
# 练习 2 参考答案
def sample_configs(space, n, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        cfg = {}
        for name, spec in space.items():
            if spec[0] == 'loguniform':
                lo, hi = spec[1], spec[2]
                cfg[name] = 10 ** rng.uniform(np.log10(lo), np.log10(hi))
            elif spec[0] == 'choice':
                cfg[name] = spec[1][rng.integers(len(spec[1]))]
        out.append(cfg)
    return out

In [ ]:
# 练习 3 参考答案
def is_reproducible(run_fn, cfg):
    r1 = run_fn(dict(cfg))
    r2 = run_fn(dict(cfg))
    return r1 == r2

---
## 🧪 真实数据胶囊：一条完整的小型研究管线

把四件事串成一条真实研究里天天用的管线：**config → sweep → 可复现多种子跑 → 聚合选最优**。
把这里的『玩具目标函数』换成『真实训练』，整套流程原样适用——这就是一个可迁移的研究工程骨架。

In [ ]:
def research_pipeline(search_space, budget, n_seeds, seed=0):
    '''完整管线：随机 sweep budget 个配置，每个跑 n_seeds 个种子，聚合后返回最优配置。'''
    configs = sample_configs(search_space, budget, seed=seed)
    records = []
    for i, sampled in enumerate(configs):
        for s in range(n_seeds):
            cfg = make_config(lr=float(sampled['lr']),
                              n_layers=int(sampled['n_layers']), seed=s)
            records.append(dict(config_id=i, lr=cfg['lr'],
                                n_layers=cfg['n_layers'], seed=s,
                                score=run_experiment(cfg)))
    df = pd.DataFrame(records)
    # 聚合：每个配置的均值±std(跨种子)，按均值选最优
    agg = (df.groupby(['config_id', 'lr', 'n_layers'])['score']
             .agg(['mean', 'std', 'count']).reset_index()
             .sort_values('mean', ascending=False))
    return df, agg

space = {'lr': ('loguniform', 1e-3, 1e-1), 'n_layers': ('choice', [2, 3, 4])}
raw, agg = research_pipeline(space, budget=8, n_seeds=5, seed=0)
print(f'总运行数 = {len(raw)} (8 配置 × 5 种子)')
print('\n聚合后 Top-3 配置(均值±std)：')
print(agg.head(3).to_string(index=False))
best = agg.iloc[0]
assert len(raw) == 40
assert (agg['count'] == 5).all(), '每个配置 5 个种子'
assert best['lr'] < 0.05, '最优 lr 应接近真实最优 0.01'
print(f"\n✅ 管线选出最优配置: lr={best['lr']:.4f}, 均值={best['mean']:.4f}±{best['std']:.4f}")

**🧪 胶囊练习**：实现 `pipeline_summary_table(agg)`：把聚合表格式化成可直接放进论文的字符串行 `'lr=X, layers=Y: ZZ.Z ± W.W (n=5)'`（百分号、保留小数）。返回 Top-k 行的列表。

In [ ]:
def pipeline_summary_table(agg, k=3):
    # TODO: 对 agg.head(k) 每行生成形如
    #   'lr=0.0100, layers=3: 90.0 ± 0.3 (n=5)'
    # 用 mean*100、std*100，返回字符串列表
    raise NotImplementedError

In [ ]:
# 自测
lines = pipeline_summary_table(agg, k=3)
assert isinstance(lines, list) and len(lines) == 3
assert all('±' in ln and 'n=5' in ln for ln in lines), '每行应含误差棒与种子数'
for ln in lines: print(ln)
print('✅ 胶囊练习通过：能把聚合结果格式化成可放进论文的行(带方差与n)')

In [ ]:
# 📖 胶囊参考答案
def pipeline_summary_table(agg, k=3):
    out = []
    for _, row in agg.head(k).iterrows():
        out.append(f"lr={row['lr']:.4f}, layers={int(row['n_layers'])}: "
                   f"{row['mean']*100:.1f} ± {row['std']*100:.1f} (n={int(row['count'])})")
    return out

### 小结
- **config 管理**：参数集中成可存档/可覆盖/可序列化的 config，杜绝硬编码 -> 可复现+可sweep的前提。
- **sweep**：网格枚举所有组合(指数爆炸)；随机搜索在少数超参重要时同预算下常更优(Bergstra & Bengio)。
- **可复现**：固定**所有**随机源(不只一个种子)；核验=同 config 两次逐位相同；记录 config+代码版本+环境。
- **聚合**：pandas groupby+agg 把多种子运行变成带方差的表；count 必报；std 就是误差棒来源。
- **最小纪律集**：参数进config、固定种子、记录每次运行、用版本控制、分离选择与评估、聚合而非堆砌。

下一站：**模块 05 · 科学写作与同行评审** —— 把可复现的结果诚实地写出来、扛住审稿。